# Optimization with SciPy: LP, MILP & Curve Fitting
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/09_Other_Experiments/scipy_optimization_linear_programming.ipynb)

Optimization answers 'what is the BEST decision?' - production mixes, routing, budget allocation, parameter fitting. SciPy covers continuous (linprog), mixed-integer (milp) and nonlinear (minimize) problems without any extra solver install.

Three classic problem shapes, solved end-to-end.

## 1. Linear programming: the furniture factory

In [ ]:
import numpy as np
from scipy.optimize import linprog

# maximize 70*chairs + 50*tables  ->  minimize negative profit
c = [-70, -50]
A_ub = [[ 3,  4],     # carpentry hours  <= 240
        [ 2,  3],     # finishing hours  <= 150
        [ 1,  2]]     # wood sheets      <= 100
b_ub = [240, 150, 100]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)],
              method="highs")
print(res.status == 0 and "OPTIMAL")
print(f"chairs = {res.x[0]:.2f}, tables = {res.x[1]:.2f}")
print(f"max profit = ${-res.fun:.2f}")

## 2. Mixed-integer: same factory, whole units only

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds

lc = LinearConstraint(np.array(A_ub), -np.inf, b_ub)
res_i = milp(c=c, constraints=lc,
             integrality=[1, 1],                       # both variables integer
             bounds=Bounds([0, 0], [np.inf, np.inf]))
print(f"chairs = {res_i.x[0]:.0f}, tables = {res_i.x[1]:.0f}, profit = ${-res_i.fun:.2f}")
print("(integer answer differs from the fractional LP optimum - rounding is not enough!)")

## 3. Nonlinear: curve fitting experimental data

In [ ]:
from scipy.optimize import curve_fit

rng = np.random.default_rng(7)
t = np.linspace(0, 5, 40)
y_meas = 2.5 * np.exp(-1.3 * t) * np.cos(6 * t) + rng.normal(0, 0.06, t.size)

def damped(t, a, b, w):
    return a * np.exp(-b * t) * np.cos(w * t)

popt, pcov = curve_fit(damped, t, y_meas, p0=[2, 1, 5])
print("fitted a,b,w =", np.round(popt, 3), "+/-", np.round(np.sqrt(np.diag(pcov)), 3))

import matplotlib.pyplot as plt
plt.scatter(t, y_meas, s=14, label="data")
plt.plot(t, damped(t, *popt), "r", label="fit")
plt.legend(); plt.title("curve_fit finds physical parameters"); plt.show()

## 4. General minimization with bounds

In [ ]:
from scipy.optimize import minimize

def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

res_n = minimize(rosenbrock, x0=[-1.2, 1.0], method="L-BFGS-B",
                 bounds=[(-3, 3), (-1, 5)])
print("minimum at:", res_n.x.round(4), "(true optimum [1, 1])")

## Choosing a tool
| Problem | Solver |
|---|---|
| linear objective + linear constraints | `linprog` (HiGHS) |
| + some variables must be integers | `milp` |
| smooth nonlinear | `minimize` (L-BFGS-B / SLSQP) |
| data fitting | `curve_fit` / `least_squares` |
| combinatorial (routing/scheduling) | Google OR-Tools (separate package) |

Always check `res.status == 0` before trusting results, and prefer HiGHS methods - they are industrial grade.